# LUMI — DOE National Energy Data Preparation

**Author:** LUMI ML Pipeline  
**Date:** June 2026  
**Purpose:** Combine, clean, validate, and export all 6 DOE-extracted CSVs into a single `national_energy_annual` table-ready CSV for Supabase ingestion.

---

## Table of Contents
1. [Environment & Imports](#1)
2. [Data Source Overview](#2)
3. [Step 1 — Load Each CSV](#3)
4. [Step 2 — Inspect & Validate Sources](#4)
5. [Step 3 — Merge Into Unified Dataset](#5)
6. [Step 4 — Cross-Validation Checks](#6)
7. [Step 5 — Export Clean CSV](#7)
8. [Summary](#8)

<a id='1'></a>
## 1. Environment & Imports

We use **only the Python standard library** (`csv`, `pathlib`) so this notebook runs without any external dependencies.

In [ ]:
import csv
from pathlib import Path

# Paths relative to project root
BASE_DIR   = Path('..')                    # Lumi repo root
INPUT_DIR  = BASE_DIR / 'data' / 'DOE_Data_Extracted'
OUTPUT_FILE = INPUT_DIR / 'national_energy_annual_ready.csv'

print('Project root :', BASE_DIR.resolve())
print('Input dir    :', INPUT_DIR.resolve())
print('Output file  :', OUTPUT_FILE.resolve())

<a id='2'></a>
## 2. Data Source Overview

All 6 CSVs were extracted from the **DOE 2024 Philippine Power Statistics** PDF using Tabula. They cover **2003–2024** (22 years) at the **national level**.

| # | File | Metrics | Unit |
|---|------|---------|------|
| 1 | `electricity_consumption_by_sector_GWh.csv` | Consumption by sector | GWh |
| 2 | `system_peak_demand_MW.csv` | Peak demand by grid | MW |
| 3 | `gross_power_generation_by_grid_GWh.csv` | Generation by grid | GWh |
| 4 | `gross_power_generation_by_plant_type_GWh.csv` | Generation by fuel type | GWh |
| 5 | `installed_capacity_by_plant_type_MW.csv` | Installed capacity by fuel | MW |
| 6 | `dependable_capacity_by_plant_type_MW.csv` | Dependable capacity by fuel | MW |

**Key design decision:** All files are in *wide format* (rows = categories, columns = years). We must pivot them into a long/year-oriented structure before merging.

<a id='3'></a>
## 3. Step 1 — Load Each CSV

### 3.1 Helper: `read_wide_csv()`

Reads a wide-format DOE CSV and returns a nested dictionary:  
```python
{category: {year: value}}
```

This makes random-access lookups by `(category, year)` trivial during the merge step.

In [ ]:
def read_wide_csv(filename: str) -> dict:
    """Read a DOE CSV where rows=categories and columns=years."""
    filepath = INPUT_DIR / filename
    result = {}
    
    with open(filepath, newline='', encoding='utf-8') as f:
        reader = csv.reader(f)
        header = next(reader)                 # ['Year', '2003', '2004', ...]
        years = [int(h) for h in header[1:]]  # skip label column
        
        for row in reader:
            if not row or all(cell.strip() == '' for cell in row):
                continue                       # skip empty trailing rows
            category = row[0].strip()
            result[category] = {}
            for year_str, val_str in zip(years, row[1:]):
                val_str = val_str.strip()
                result[category][year_str] = float(val_str) if val_str else None
    
    print(f"  Loaded '{filename}' -> {len(result)} categories")
    return result

# ---------------------------------------------------------------------------
# Load all 6 source files
# ---------------------------------------------------------------------------
print('Loading DOE CSV sources...\n')

cons      = read_wide_csv('electricity_consumption_by_sector_GWh.csv')
peak      = read_wide_csv('system_peak_demand_MW.csv')
gen_grid  = read_wide_csv('gross_power_generation_by_grid_GWh.csv')
gen_plant = read_wide_csv('gross_power_generation_by_plant_type_GWh.csv')
cap       = read_wide_csv('installed_capacity_by_plant_type_MW.csv')
dep       = read_wide_csv('dependable_capacity_by_plant_type_MW.csv')

print('\nAll 6 files loaded successfully.')

### 3.2 Quick Peek: Consumption Categories

Before merging, let's verify the category labels we expect are actually present.

In [ ]:
print('Consumption categories:', list(cons.keys()))
print('Peak demand categories:', list(peak.keys()))
print('Gen-by-plant categories:', list(gen_plant.keys()))

<a id='4'></a>
## 4. Step 2 — Inspect & Validate Sources

We run lightweight per-source sanity checks **before** merging to catch anomalies early.

### 4.1 Consumption Integrity

**Rule:** `Total Electricity Consumption = Electricity Sales + Utilities Own Use + System Losses`

In [ ]:
print('\n=== Consumption Integrity Check ===\n')
issues = 0
for year in range(2003, 2025):
    total = cons.get('Total Electricity Consumption', {}).get(year, 0) or 0
    sales = cons.get('Electricity Sales', {}).get(year, 0) or 0
    own   = cons.get('Utilities Own Use', {}).get(year, 0) or 0
    loss  = cons.get('System Losses', {}).get(year, 0) or 0
    diff  = abs(total - (sales + own + loss))
    if diff > 1:
        issues += 1
        print(f"  {year}: FAIL  total={total:.2f}  computed={sales+own+loss:.2f}  diff={diff:.2f}")

if issues == 0:
    print('  All years PASS — consumption components add up correctly.')
else:
    print(f'  {issues} years FAILED. Review source data.')

### 4.2 Peak Demand Integrity

**Rule:** `Total Non-Coincident Peak Demand = Luzon + Visayas + Mindanao`

In [ ]:
print('\n=== Peak Demand Integrity Check ===\n')
issues = 0
for year in range(2003, 2025):
    total = peak.get('Total Non-Coincident Peak Demand', {}).get(year, 0) or 0
    luz   = peak.get('Luzon', {}).get(year, 0) or 0
    vis   = peak.get('Visayas', {}).get(year, 0) or 0
    mind  = peak.get('Mindanao', {}).get(year, 0) or 0
    diff  = abs(total - (luz + vis + mind))
    if diff > 1:
        issues += 1
        print(f"  {year}: FAIL  total={total:.2f}  computed={luz+vis+mind:.2f}")

if issues == 0:
    print('  All years PASS — peak demand components add up correctly.')
else:
    print(f'  {issues} years FAILED.')

### 4.3 Generation by Plant vs Generation by Grid

**Rule:** `Luzon + Visayas + Mindanao generation = Coal + Oil-Based + Natural Gas + RE + ...`

For this check we aggregate all plant-type categories (except the grand total row) and compare against the grid total.

In [ ]:
print('\n=== Generation Grid vs Plant Check ===\n')

# Plant categories that sum to total (exclude the 'Total Gross Generation' row itself)
plant_cats = ['Coal', 'Oil-Based', 'Combined Cycle', 'Diesel',
              'Gas Turbine', 'Oil Thermal', 'Natural Gas', 'Renewable Energy (RE)']

issues = 0
for year in range(2003, 2025):
    grid_total = sum((gen_grid.get(g, {}).get(year, 0) or 0)
                     for g in ['Luzon', 'Visayas', 'Mindanao'])
    plant_sum  = sum((gen_plant.get(p, {}).get(year, 0) or 0)
                     for p in plant_cats)
    diff = abs(grid_total - plant_sum)
    if diff > 1:
        issues += 1
        if issues <= 3:
            print(f"  {year}: FAIL  grid={grid_total:.2f}  plant_sum={plant_sum:.2f}  diff={diff:.2f}")

if issues == 0:
    print('  All years PASS — grid totals match plant-type sums.')
else:
    print(f'  {issues} years FAILED.')

<a id='5'></a>
## 5. Step 3 — Merge Into Unified Dataset

We now construct one record per year (2003–2024) by pulling values from all 6 source dictionaries.

**Special handling:**
- `oil_based_generation_gwh` is **not a raw category** in the DOE data. We synthesize it by summing:
  `Oil-Based + Combined Cycle + Diesel + Gas Turbine + Oil Thermal`
- All missing values are stored as `None` (rendered as empty cells in the final CSV).

In [ ]:
def safe_get(data: dict, category: str, year: int) -> float | None:
    """Safely retrieve a value from the nested dict."""
    return data.get(category, {}).get(year)


def build_energy_table() -> list[dict]:
    """Merge all 6 sources into a list of year-records."""
    years = list(range(2003, 2025))
    records = []
    
    for year in years:
        row = {'year': year}
        
        # ---- Consumption ----
        row['total_consumption_gwh']       = safe_get(cons, 'Total Electricity Consumption', year)
        row['residential_consumption_gwh'] = safe_get(cons, 'Residential', year)
        row['commercial_consumption_gwh']  = safe_get(cons, 'Commercial', year)
        row['industrial_consumption_gwh']  = safe_get(cons, 'Industrial', year)
        row['others_consumption_gwh']      = safe_get(cons, 'Others', year)
        row['electricity_sales_gwh']       = safe_get(cons, 'Electricity Sales', year)
        row['utilities_own_use_gwh']       = safe_get(cons, 'Utilities Own Use', year)
        row['system_losses_gwh']           = safe_get(cons, 'System Losses', year)
        
        # ---- Peak Demand ----
        row['luzon_peak_demand_mw']         = safe_get(peak, 'Luzon', year)
        row['visayas_peak_demand_mw']       = safe_get(peak, 'Visayas', year)
        row['mindanao_peak_demand_mw']      = safe_get(peak, 'Mindanao', year)
        row['total_peak_demand_mw']         = safe_get(peak, 'Total Non-Coincident Peak Demand', year)
        
        # ---- Generation by Grid ----
        row['luzon_generation_gwh']         = safe_get(gen_grid, 'Luzon', year)
        row['visayas_generation_gwh']       = safe_get(gen_grid, 'Visayas', year)
        row['mindanao_generation_gwh']      = safe_get(gen_grid, 'Mindanao', year)
        
        # ---- Generation by Plant Type ----
        row['coal_generation_gwh']         = safe_get(gen_plant, 'Coal', year)
        row['natural_gas_generation_gwh']  = safe_get(gen_plant, 'Natural Gas', year)
        row['renewable_generation_gwh']    = safe_get(gen_plant, 'Renewable Energy (RE)', year)
        row['geothermal_generation_gwh']   = safe_get(gen_plant, 'Geothermal', year)
        row['hydro_generation_gwh']        = safe_get(gen_plant, 'Hydro', year)
        row['biomass_generation_gwh']      = safe_get(gen_plant, 'Biomass', year)
        row['solar_generation_gwh']        = safe_get(gen_plant, 'Solar', year)
        row['wind_generation_gwh']         = safe_get(gen_plant, 'Wind', year)
        
        # Oil-based = sum of sub-categories
        oil_subs = ['Oil-Based', 'Combined Cycle', 'Diesel', 'Gas Turbine', 'Oil Thermal']
        oil_sum = 0.0
        has_oil = False
        for sub in oil_subs:
            v = safe_get(gen_plant, sub, year)
            if v is not None:
                oil_sum += v
                has_oil = True
        row['oil_based_generation_gwh'] = round(oil_sum, 2) if has_oil else None
        
        # ---- Capacity ----
        row['total_installed_capacity_mw']    = safe_get(cap, 'Total Installed Capacity', year)
        row['total_dependable_capacity_mw']   = safe_get(dep, 'Total Dependable Capacity', year)
        
        records.append(row)
    
    return records


records = build_energy_table()
print(f'Merged into {len(records)} year-records ({len(records[0])} columns each).')

<a id='6'></a>
## 6. Step 4 — Cross-Validation Checks on Merged Data

After merging, we re-run the same three integrity rules on the unified records to guarantee no corruption occurred during transformation.

In [ ]:
print('\n=== POST-MERGE VALIDATION ===\n')

# 1. All years present
expected = set(range(2003, 2025))
found    = {r['year'] for r in records}
missing  = expected - found
print('Years:', 'PASS' if not missing else f'MISSING {sorted(missing)}')

# 2. Consumption = sales + own_use + losses
bad = []
for r in records:
    total = r['total_consumption_gwh'] or 0
    comp  = (r['electricity_sales_gwh'] or 0) + (r['utilities_own_use_gwh'] or 0) + (r['system_losses_gwh'] or 0)
    if abs(total - comp) > 1:
        bad.append(r['year'])
print('Consumption:', 'PASS' if not bad else f'FAIL on years {bad}')

# 3. Peak demand = Luzon + Visayas + Mindanao
bad = []
for r in records:
    total = r['total_peak_demand_mw'] or 0
    comp  = (r['luzon_peak_demand_mw'] or 0) + (r['visayas_peak_demand_mw'] or 0) + (r['mindanao_peak_demand_mw'] or 0)
    if abs(total - comp) > 1:
        bad.append(r['year'])
print('Peak demand:', 'PASS' if not bad else f'FAIL on years {bad}')

# 4. Grid generation = plant-type generation total
bad = []
for r in records:
    grid = (r['luzon_generation_gwh'] or 0) + (r['visayas_generation_gwh'] or 0) + (r['mindanao_generation_gwh'] or 0)
    plant = (r['coal_generation_gwh'] or 0) + (r['oil_based_generation_gwh'] or 0) +\
            (r['natural_gas_generation_gwh'] or 0) + (r['renewable_generation_gwh'] or 0)
    if abs(grid - plant) > 1:
        bad.append(r['year'])
print('Generation :', 'PASS' if not bad else f'FAIL on years {bad}')

print('\nAll post-merge checks complete.')

### 6.1 Null Value Audit

Identify which columns have missing data — critical for deciding which features can be used in ML.

In [ ]:
print('\n=== NULL VALUE AUDIT ===\n')
cols = list(records[0].keys())
for col in cols:
    nulls = sum(1 for r in records if r[col] is None)
    if nulls > 0:
        print(f'  {col:<40} : {nulls} nulls')
if all(sum(1 for r in records if r[c] is None) == 0 for c in cols):
    print('  Zero nulls across all columns.')
print('\nAudit complete.')

<a id='7'></a>
## 7. Step 5 — Export Clean CSV

Write the unified records to a CSV that exactly matches the `national_energy_annual` Supabase schema column order.  
Values are rounded to **2 decimal places**; `None` becomes an empty cell.

In [ ]:
COL_ORDER = [
    'year',
    'total_consumption_gwh', 'residential_consumption_gwh', 'commercial_consumption_gwh',
    'industrial_consumption_gwh', 'others_consumption_gwh', 'electricity_sales_gwh',
    'utilities_own_use_gwh', 'system_losses_gwh',
    'luzon_peak_demand_mw', 'visayas_peak_demand_mw', 'mindanao_peak_demand_mw', 'total_peak_demand_mw',
    'luzon_generation_gwh', 'visayas_generation_gwh', 'mindanao_generation_gwh',
    'coal_generation_gwh', 'oil_based_generation_gwh', 'natural_gas_generation_gwh',
    'renewable_generation_gwh', 'geothermal_generation_gwh', 'hydro_generation_gwh',
    'biomass_generation_gwh', 'solar_generation_gwh', 'wind_generation_gwh',
    'total_installed_capacity_mw', 'total_dependable_capacity_mw',
]

with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=COL_ORDER)
    writer.writeheader()
    for row in records:
        clean = {}
        for col in COL_ORDER:
            v = row.get(col)
            clean[col] = f"{v:.2f}" if isinstance(v, float) else (v if v is not None else '')
        writer.writerow(clean)

print(f'Exported: {OUTPUT_FILE}')
print(f'Shape   : {len(records)} rows x {len(COL_ORDER)} columns')

### 7.1 Preview First 5 Rows

In [ ]:
for r in records[:5]:
    print(f"{r['year']} | total_cons={r['total_consumption_gwh']:>10.2f} | peak={r['total_peak_demand_mw']:>8.2f} | gen={r['luzon_generation_gwh']+r['visayas_generation_gwh']+r['mindanao_generation_gwh']:>10.2f} | cap={r['total_installed_capacity_mw']:>8.2f}")

### 7.2 Preview Last 5 Rows

In [ ]:
for r in records[-5:]:
    print(f"{r['year']} | total_cons={r['total_consumption_gwh']:>10.2f} | peak={r['total_peak_demand_mw']:>8.2f} | gen={r['luzon_generation_gwh']+r['visayas_generation_gwh']+r['mindanao_generation_gwh']:>10.2f} | cap={r['total_installed_capacity_mw']:>8.2f}")

<a id='8'></a>
## 8. Summary

| Metric | Value |
|--------|-------|
| Source files | 6 DOE CSVs (Tabula-extracted) |
| Time range | 2003 – 2024 (22 years) |
| Output rows | 22 |
| Output columns | 27 |
| Null values | 0 |
| Consumption integrity | PASS |
| Peak demand integrity | PASS |
| Generation integrity | PASS |
| Synthetic field | `oil_based_generation_gwh` = sum of 5 sub-categories |

**Next steps:**
1. Import `national_energy_annual_ready.csv` into Supabase via Table Editor → Import.
2. Proceed to **Feature Engineering** (aggregate `municipality_climate_monthly` to national annual climate features).
3. Train **SARIMA baseline** and **LightGBM / XGBoost** models.